In [ ]:
import rasterio as rio
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.mask import mask
import numpy as np
from shapely.geometry import box
import math
from collections import defaultdict
from numpy import fft
import os
import cv2
from skimage import io, feature, exposure
import pywt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.utils.data import DataLoader, random_split
from sklearn.preprocessing import normalize
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from skimage.feature import peak_local_max
from scipy.signal import find_peaks
from torchvision import models
import torch.optim as optim
from sklearn.metrics import f1_score
from tqdm import tqdm
from torch.utils.data import TensorDataset
from texture_encoders import *
from utils import *
from MLP import *

class_dict ={
    0.0:'Cultivated',
    1.0:'Uprooted',
    2.0:'Abandoned',
}

class_images = defaultdict(list)
cmap = "Greens"

# Load data

In [ ]:
year=... # example : 2023
nb_splits = ... # example : 3
image_dir='data/image'
gdf_path='data/polygons.gpkg'
index_col = 'id'
class_col = 'label'
output_path = 'data'
id_path=''


gdf = gpd.read_file(gdf_path).set_index(index_col)

## Split data

In [ ]:
all_ids = sorted([
    int(os.path.splitext(f)[0]) 
    for f in os.listdir(image_dir) 
    if f.lower().endswith(".jpg")
])

labels = [int(gdf.loc[obj_id][class_col]) for obj_id in all_ids]

split_size_tab = pd.DataFrame(columns=['Train', 'Validation', 'Test'])
for split in range(nb_splits):
    train_ids, temp_ids, train_labels, temp_labels = train_test_split(
        all_ids, labels, train_size=0.4, stratify=labels
    )
    val_ids, test_ids = train_test_split(
        temp_ids, train_size=0.6, stratify=temp_labels
    )

    split_size_tab.loc[split, 'Train'] = len(train_ids)
    split_size_tab.loc[split, 'Validation'] = len(val_ids)
    split_size_tab.loc[split, 'Test'] = len(test_ids)

    train_dataset_raw = TextureDataset(image_dir, gdf_path, id_list=train_ids)
    train_mean, train_std = compute_mean_std(train_dataset_raw)
    
    np.save(f"{output_path}/split_{split+1}/train_mean.npy", np.array([train_mean]))
    np.save(f"{output_path}/split_{split+1}/train_std.npy",  np.array([train_std]))

    if not os.path.exists(f'{output_path}/split_{split+1}'):
        os.makedirs(f'{output_path}/split_{split+1}')

    np.save(f"{output_path}/split_{split+1}/train_ids.npy", np.array(train_ids))
    np.save(f"{output_path}/split_{split+1}/val_ids.npy",   np.array(val_ids))
    np.save(f"{output_path}/split_{split+1}/test_ids.npy",  np.array(test_ids))

split_size_tab

# Self-supervised auto-encoding

## Compute and save features

In [ ]:
for split in range(nb_splits):
    autoencoder_pth = f"{output_path}/split_{split+1}/autoencoder.pt"


    train_ids = np.load(f"{output_path}/split_{split+1}/train_ids.npy")
    val_ids = np.load(f"{output_path}/split_{split+1}/val_ids.npy")
    test_ids = np.load(f"{output_path}/split_{split+1}/test_ids.npy")


    train_mean = np.load(f"{output_path}/split_{split+1}/train_mean.npy")
    train_std = np.load(f"{output_path}/split_{split+1}/train_std.npy")


    model = train_autoencoder(
        image_dir, gdf_path,
        train_ids, train_mean, train_std,
        save_path=autoencoder_pth
    )


    # Extract features for all three splits
    df_train = extract_features(image_dir, gdf_path, train_ids,
                                train_mean, train_std,
                                model_path=autoencoder_pth)
    df_val   = extract_features(image_dir, gdf_path, val_ids,
                                train_mean, train_std,
                                model_path=autoencoder_pth)
    df_test  = extract_features(image_dir, gdf_path, test_ids,
                                train_mean, train_std,
                                model_path=autoencoder_pth)
    
    if not os.path.exists(f'{output_path}/split_{split+1}/'):
        os.mkdir(f'{output_path}/split_{split+1}/')
    
    df_train.to_pickle(f"{output_path}/split_{split+1}/train_texture_features.pkl")
    df_val.to_pickle(f"{output_path}/split_{split+1}/val_texture_features.pkl")
    df_test.to_pickle(f"{output_path}/split_{split+1}/test_texture_features.pkl")
    


## Classify based on features

In [ ]:
data_sources = [] # example : [s1, s2]

for split in range(nb_splits):

    cr_dict = train_test_split(
        output_path,
        id_path='',
        split=split,
        aux_data_source_pths=['s1', 's2']

    )
